# 177 — Attention Capacity Analysis

Analyze attention head capacity: pattern rank, information throughput,
bottleneck detection, head utilization, and capacity allocation.

## Why This Matters

Attention capacity reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_capacity_analysis import (
    attention_pattern_rank,
    attention_information_throughput,
    attention_bottleneck_detection,
    head_utilization,
    capacity_allocation,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = attention_pattern_rank(model, tokens, layer=0)
print(f"Mean effective rank: {result['mean_effective_rank']:.2f}\n")
for h in result['per_head']:
    print(f"Head {h['head']}: eff_rank={h['effective_rank']:.2f}  "
          f"ratio={h['rank_ratio']:.3f}  sig={h['n_significant']}")

In [ ]:
result = attention_information_throughput(model, tokens, layer=0)
print(f"Mean throughput: {result['mean_throughput']:.3f}\n")
for h in result['per_head']:
    bar = '#' * int(h['throughput'] * 20)
    print(f"Head {h['head']}: throughput={h['throughput']:.3f}  "
          f"entropy={h['mean_entropy']:.3f}  {bar}")

In [ ]:
result = attention_bottleneck_detection(model, tokens)
print(f"Most bottlenecked layer: {result['most_bottlenecked_layer']}\n")
for layer in result['per_layer']:
    print(f"Layer {layer['layer']}: bottleneck={layer['bottleneck_score']:.3f}  "
          f"mean_rank={layer['mean_effective_rank']:.2f}  "
          f"min_rank={layer['min_effective_rank']:.2f}")

In [ ]:
result = head_utilization(model, tokens, layer=0)
print(f"Mean utilization: {result['mean_utilization']:.3f}\n")
for h in result['per_head']:
    bar = '#' * int(h['utilization'] * 30)
    print(f"Head {h['head']}: util={h['utilization']:.3f}  "
          f"output_norm={h['output_norm']:.4f}  {bar}")

In [ ]:
result = capacity_allocation(model, tokens)
print(f"Total capacity: {result['total_capacity']:.3f}  Gini: {result['gini_coefficient']:.3f}")
print(f"Most active: L{result['most_active_head']['layer']}H{result['most_active_head']['head']}")
print(f"\nLayer shares:")
for i, share in enumerate(result['layer_share']):
    bar = '#' * int(share * 60)
    print(f"  Layer {i}: {share:.3f}  {bar}")